# 03. 서브셋 추가 뽑기 (검토에서 빠진 만큼 채우기)

검토(02)에서 bad·fix로 빠진 만큼 **같은 후보 3,000장, 같은 기준**으로 사진을 더 뽑아 `subset/`에 붙인다.

- 기존 498장은 그대로 둔다. 기존 사진, 그리고 기존 사진과 비슷한 그림(중복 그룹)은 뽑지 않는다.
- 그룹(clean/shadow/complex)과 dev/test마다 **원래 장수 − 남은 ok 수**만큼, 검토에서 빠질 비율을 감안해 넉넉히 뽑는다.
- 새 사진은 `dev_full.json`, `test_full.json`에 `batch=2`로 추가된다. `dev.json`, `test.json`은 02 노트북 5번 셀이 다시 만든다.
- 다음 순서: 이 노트북 → 02 노트북에서 `ONLY_NEW = True`로 새 사진만 검토 → 02의 4번(합치기), 5번(반영).

**한 명만 한 번 실행한다.** 다시 실행해도 이미 추가된 상태면 아무것도 바꾸지 않는다.

In [ ]:
# 1. 설정
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os, sys, json, time, math, shutil, tarfile, urllib.request, unicodedata
import numpy as np, pandas as pd, cv2

FOLDER = '컴퓨터비전'                              # 팀 공유 폴더 (subset/이 있는 곳)
BASE = None                                       # 자동으로 못 찾으면 직접 적는다
ORIG_DIR = '/content/drive/MyDrive/cv_project/data'   # 01 노트북을 처음 돌린 곳 (어노테이션 JSON, pool_features.csv)
POOL_DIR = '/content/ad_pool'                     # 후보 사진은 Colab 로컬 디스크에 받는다

# 01 노트북과 같아야 같은 후보·같은 그룹 기준이 나온다 (바꾸지 않는다)
POOL_N, SEED = 3000, 42
CONDS = ['clean', 'shadow', 'complex']
ADD_SEED = 43        # 추가 뽑기 순서 (처음 뽑기와 다르게)
MARGIN = 1.15        # 검토에서 빠질 것까지 감안해 더 뽑는 배율 (1.0이면 딱 예상치만큼)

ANN_URL = 'https://dl.fbaipublicfiles.com/amateur_drawings/amateur_drawings_annotations.json'
TAR_URL = 'https://dl.fbaipublicfiles.com/amateur_drawings/amateur_drawings.tar'


def nfc(s):
    return unicodedata.normalize('NFC', s)


def find_base(name, roots=('/content/drive/MyDrive', '/content/drive/Shareddrives')):
    for root in roots:
        for d, dirs, _ in os.walk(root):
            for x in dirs:
                if nfc(x) == nfc(name) and os.path.isdir(os.path.join(d, x, 'subset')):
                    return os.path.join(d, x)
            if d[len(root):].count('/') >= 1:
                dirs[:] = []
    return None


BASE = BASE or find_base(FOLDER)
assert BASE, f"'{FOLDER}' 폴더를 못 찾음: 내 드라이브에 바로가기가 있는지 확인"
SUB = os.path.join(BASE, 'subset')
os.makedirs(POOL_DIR, exist_ok=True)
sys.path.insert(0, BASE)
from ad_utils import load_image, ann_to_mask, bbox_int
print('BASE =', BASE)

In [ ]:
# 2. 어노테이션 읽기 + 후보 3,000장 다시 받기 (tar 앞부분이라 01과 같은 3,000장이 나온다, 약 10분)
ANN_PATH = os.path.join(ORIG_DIR, 'amateur_drawings_annotations.json')
if not os.path.exists(ANN_PATH):
    ANN_PATH = '/content/amateur_drawings_annotations.json'
    if not os.path.exists(ANN_PATH):
        urllib.request.urlretrieve(ANN_URL, ANN_PATH)
with open(ANN_PATH) as f:
    coco = json.load(f)
imgs = {im['id']: im for im in coco['images']}
anns = {a['image_id']: a for a in coco['annotations']}
by_name = {os.path.basename(im['file_name']): im['id'] for im in coco['images']}

rows = []
for a in coco['annotations']:
    im = imgs[a['image_id']]
    W, H = im['width'], im['height']
    s = min(1.0, 512 / max(W, H))
    x, y, w, h = a['bbox']
    rows.append(dict(image_id=a['image_id'], orig_w=W, orig_h=H, img_w=round(W * s), img_h=round(H * s),
                     bx=x, by=y, bw=w, bh=h, area=a['area'], n_poly=len(a['segmentation'])))
meta = pd.DataFrame(rows)
meta['bbox_frac'] = meta.bw * meta.bh / (meta.img_w * meta.img_h)
meta['fill'] = meta.area / (meta.bw * meta.bh).clip(lower=1)
meta['full_bbox'] = (meta.bx <= 1) & (meta.by <= 1) & (meta.bw >= meta.img_w - 2) & (meta.bh >= meta.img_h - 2)


def stream_pool(n, out_dir):
    have = {f for f in os.listdir(out_dir) if f in by_name}
    if len(have) >= n:
        print(f'이미 {len(have)}장 있음 → 건너뜀'); return
    t = time.time(); got = len(have)
    with urllib.request.urlopen(TAR_URL) as r, tarfile.open(fileobj=r, mode='r|gz') as tf:
        for m in tf:
            if not (m.isfile() and m.name.endswith('.png')): continue
            fn = os.path.basename(m.name)
            if fn not in by_name or fn in have: continue
            with open(os.path.join(out_dir, fn), 'wb') as f:
                f.write(tf.extractfile(m).read())
            got += 1
            if got % 500 == 0: print(f'  {got}/{n}  ({time.time()-t:.0f}s)')
            if got >= n: break
    print(f'완료: {got}장, {time.time()-t:.0f}s')


stream_pool(POOL_N, POOL_DIR)
pool_ids = sorted(by_name[f] for f in os.listdir(POOL_DIR) if f in by_name)
def pool_path(iid): return os.path.join(POOL_DIR, os.path.basename(imgs[iid]['file_name']))
print('후보 풀:', len(pool_ids), '장')

In [ ]:
# 3. 촬영 조건 그룹과 품질 필터 (01 노트북 6·7번과 같은 기준)
def image_features(iid):
    img = load_image(pool_path(iid)); h, w = img.shape[:2]
    gt = ann_to_mask(anns[iid], h, w)
    k = max(3, int(0.03 * max(h, w)) | 1)
    bg = cv2.dilate(gt, np.ones((k, k), np.uint8)) == 0
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB); L = lab[:, :, 0].astype(np.float32)
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    f = dict(image_id=iid, h=h, w=w, fg_frac=gt.mean(), bg_frac=bg.mean())
    if bg.sum() < 0.05 * h * w:
        f.update(bg_bright=np.nan, bg_uneven=np.nan, bg_edges=np.nan, bg_sat=np.nan, fg_contrast=np.nan)
    else:
        cells = []
        for ys in np.array_split(np.arange(h), 4):
            for xs in np.array_split(np.arange(w), 4):
                c = bg[ys[0]:ys[-1]+1, xs[0]:xs[-1]+1]
                if c.sum() > 0.2 * c.size:
                    cells.append(np.median(L[ys[0]:ys[-1]+1, xs[0]:xs[-1]+1][c]))
        edges = cv2.Canny(cv2.GaussianBlur(gray, (3, 3), 0), 50, 150) > 0
        f.update(bg_bright=np.median(L[bg]),
                 bg_uneven=(max(cells) - min(cells)) if len(cells) >= 2 else np.nan,
                 bg_edges=edges[bg].mean(),
                 bg_sat=np.median(hsv[:, :, 1][bg]),
                 fg_contrast=np.median(L[bg]) - np.median(L[gt > 0]) if gt.sum() else np.nan)
    f['sharpness'] = cv2.Laplacian(gray, cv2.CV_64F).var()
    f['touch_border'] = bool(gt[0].any() or gt[-1].any() or gt[:, 0].any() or gt[:, -1].any())
    n_cc, _ = cv2.connectedComponents(gt)
    f['n_parts'] = n_cc - 1
    return f


feat_path = os.path.join(ORIG_DIR, 'pool_features.csv')
if os.path.exists(feat_path) and sorted(pd.read_csv(feat_path).image_id) == pool_ids:
    feat = pd.read_csv(feat_path); print('pool_features.csv 재사용')
else:
    t = time.time(); feat = pd.DataFrame([image_features(i) for i in pool_ids]); print(f'특징 계산 {time.time()-t:.0f}s')
feat = feat.merge(meta[['image_id', 'orig_w', 'orig_h', 'bbox_frac', 'fill', 'full_bbox', 'n_poly']], on='image_id')

thr = dict(edge=feat.bg_edges.quantile(0.75), uneven=feat.bg_uneven.quantile(0.70), dark=feat.bg_bright.quantile(0.15))
def condition(r):
    if pd.isna(r.bg_edges): return 'no_bg'
    if r.bg_edges >= thr['edge']: return 'complex'
    if r.bg_uneven >= thr['uneven'] or r.bg_bright <= thr['dark']: return 'shadow'
    return 'clean'
feat['condition'] = feat.apply(condition, axis=1)
feat['usable'] = ~(feat.full_bbox | (feat.condition == 'no_bg') | (feat.fill < 0.05)
                   | feat.image_id.map(lambda i: anns[i]['area'] <= 0) | (feat.n_parts > 3) | (feat.bbox_frac < 0.05))
print('사용 가능:', int(feat.usable.sum()), '/', len(feat))

In [ ]:
# 4. 비슷한 그림 묶기 (01 노트북과 같은 방법)
def crop_gray(iid):
    img = load_image(pool_path(iid)); h, w = img.shape[:2]
    x0, y0, x1, y1 = bbox_int(anns[iid], h, w)
    return cv2.cvtColor(img[y0:y1, x0:x1], cv2.COLOR_RGB2GRAY)

def dhash(g, size=8):
    g = cv2.resize(g, (size + 1, size), interpolation=cv2.INTER_AREA)
    return (g[:, 1:] > g[:, :-1]).ravel()

def thumb(g, size=16):
    t = cv2.resize(g, (size, size), interpolation=cv2.INTER_AREA).astype(np.float32).ravel()
    return (t - t.mean()) / (t.std() + 1e-6)

usable_ids = feat[feat.usable].image_id.tolist()
crops = [crop_gray(i) for i in usable_ids]
bits = np.stack([dhash(g) for g in crops]).astype(np.int16)
thumbs = np.stack([thumb(g) for g in crops])
dist = (bits[:, None, :] != bits[None, :, :]).sum(-1)
informative = (bits.sum(1) >= 10) & (bits.sum(1) <= 54)
parent = list(range(len(usable_ids)))
def find(x):
    while parent[x] != x: parent[x] = parent[parent[x]]; x = parent[x]
    return x
cand_pairs = np.argwhere(np.triu(dist <= 6, 1))
pairs = [(a, b) for a, b in cand_pairs
         if informative[a] and informative[b] and float(thumbs[a] @ thumbs[b]) / thumbs.shape[1] > 0.9]
for a, b in pairs: parent[find(a)] = find(b)
feat['dup_group'] = feat.image_id.map({iid: int(find(k)) for k, iid in enumerate(usable_ids)})
print('유사 쌍', len(pairs), '개')

In [ ]:
# 5. 지금 상태와 더 뽑을 장수 계산
def load_full(sp):
    p = os.path.join(SUB, f'{sp}_full.json')
    if not os.path.exists(p):                      # 02의 5번 셀을 아직 안 돌렸으면 지금 json이 원본
        shutil.copy(os.path.join(SUB, f'{sp}.json'), p)
    keep = os.path.join(SUB, f'{sp}_full_498.json')  # 처음 뽑은 상태 보관 (한 번만)
    if not os.path.exists(keep):
        shutil.copy(p, keep)
    return json.load(open(p))

full = {sp: load_full(sp) for sp in ['dev', 'test']}
ALREADY = any(im.get('batch') == 2 for sp in full for im in full[sp]['images'])
existing = {im['id']: (sp, im['condition'], im.get('batch', 1)) for sp in full for im in full[sp]['images']}

# 그룹 기준이 01과 같은지 확인 (100%여야 정상)
same = np.mean([feat.set_index('image_id').condition.get(i) == c for i, (_, c, _) in existing.items()])
print(f'그룹 일치율 {same*100:.1f}%  (100%가 아니면 POOL_N/SEED가 01과 다른 것)')
assert same > 0.99, '그룹 기준이 01 노트북과 다름 → 멈춤'

rev = pd.read_csv(os.path.join(SUB, 'review_all.csv'))
status = dict(zip(rev.image_id, rev.status))
plan = []
for sp in ['dev', 'test']:
    for c in CONDS:
        b1 = [i for i, (s, cc, b) in existing.items() if s == sp and cc == c and b == 1]
        ok = sum(status.get(i) == 'ok' for i in b1)
        rate = ok / len(b1) if b1 else 1
        need = len(b1) - ok
        plan.append(dict(split=sp, condition=c, target=len(b1), ok=ok, ok_rate=round(rate, 2), need=need,
                         draw=math.ceil(need / max(rate, 0.3) * MARGIN) if need > 0 else 0))
plan = pd.DataFrame(plan)
print(plan.to_string(index=False))
print('\n새로 뽑을 사진 합계:', plan.draw.sum(), '장  (4명이면 1인당 약', math.ceil(plan.draw.sum() / 4), '장)')

In [ ]:
# 6. 뽑기: 기존 사진과 그 유사 그룹은 제외, 유사 그룹당 1장
if ALREADY:
    print('이미 추가된 상태 → 새로 뽑지 않음'); add = pd.DataFrame()
else:
    used_groups = set(feat[feat.image_id.isin(existing)].dup_group.dropna())
    cand = feat[feat.usable & feat.condition.isin(CONDS) & ~feat.image_id.isin(existing) & ~feat.dup_group.isin(used_groups)]
    cand = cand.sample(frac=1, random_state=ADD_SEED).drop_duplicates('dup_group')
    picked = []
    for c in CONDS:
        g = cand[cand.condition == c]
        n_test = int(plan[(plan.split == 'test') & (plan.condition == c)].draw.iloc[0])
        n_dev = int(plan[(plan.split == 'dev') & (plan.condition == c)].draw.iloc[0])
        if len(g) < n_test + n_dev:
            print(f'⚠️ {c}: 후보 {len(g)}장 < 필요 {n_test + n_dev}장 → test부터 채움')
        t = g.iloc[:n_test].assign(split='test'); d = g.iloc[n_test:n_test + n_dev].assign(split='dev')
        picked += [t, d]
    add = pd.concat(picked)
    add['pick_rank'] = range(len(add))           # 나중에 장수를 맞출 때 앞 순서부터 쓴다
    print(pd.crosstab(add.condition, add.split, margins=True))

In [ ]:
# 7. 저장: 사진·마스크 복사, *_full.json에 batch=2로 추가, 목록 CSV
if not ALREADY and len(add):
    img_out, mask_out = os.path.join(SUB, 'images'), os.path.join(SUB, 'masks_gt')
    for sp in ['dev', 'test']:
        out = full[sp]
        for im in out['images']:
            im.setdefault('batch', 1)
        for r in add[add.split == sp].itertuples():
            src = imgs[r.image_id]; fn = os.path.basename(src['file_name'])
            shutil.copy(pool_path(r.image_id), os.path.join(img_out, fn))
            mfn = fn.replace('.png', '_mask.png')
            cv2.imwrite(os.path.join(mask_out, mfn), ann_to_mask(anns[r.image_id], int(r.h), int(r.w)) * 255)
            out['images'].append(dict(id=int(r.image_id), file_name=f'images/{fn}', mask_file=f'masks_gt/{mfn}',
                                      width=int(r.w), height=int(r.h), orig_width=src['width'], orig_height=src['height'],
                                      condition=r.condition, license=src['license'], batch=2, pick_rank=int(r.pick_rank)))
            out['annotations'].append(anns[r.image_id])
        with open(os.path.join(SUB, f'{sp}_full.json'), 'w') as f:
            json.dump(out, f)
    cols = ['image_id', 'split', 'condition', 'dup_group', 'h', 'w', 'orig_w', 'orig_h', 'bg_bright', 'bg_uneven',
            'bg_edges', 'bg_sat', 'fg_contrast', 'sharpness', 'fill', 'bbox_frac', 'n_parts', 'pick_rank']
    add.assign(file_name=add.image_id.map(lambda i: 'images/' + os.path.basename(imgs[i]['file_name'])))[['file_name'] + cols] \
       .to_csv(os.path.join(SUB, 'add_list.csv'), index=False)
    print('추가 완료:', len(add), '장 → 다음: 02 노트북에서 ONLY_NEW = True로 새 사진 검토')
n = {sp: len(json.load(open(os.path.join(SUB, f'{sp}_full.json')))['images']) for sp in ['dev', 'test']}
print('현재 *_full.json:', n)